# 20.11 多任务学习 / Multi-task Learning (MTL)

**中文**:通常我们**一个模型解决一个任务**。**多任务学习(Multi-task Learning, MTL)** 反其道:**一个模型同时学多个相关任务,共享底层表示**。动机:①**归纳迁移**——任务 A 学到的特征能帮任务 B(相关任务互为"正则化",提升泛化);②**数据高效**——数据少的任务借助数据多的相关任务(共享表示≈变相扩充数据);③**省资源**——一个模型代替 N 个(手机上、推荐系统里,一次前向出多个预测);④生物合理性(人也是一脑多用)。经典结构:**硬参数共享**(共享主干 + 各任务专属输出头)。但 MTL **绝不是免费午餐**——本节从零实现,并诚实揭示它成立的条件与**负迁移(negative transfer)** 的陷阱。
**English**: Usually **one model solves one task**. **Multi-task Learning (MTL)** does the opposite: **one model learns several related tasks at once, sharing the underlying representation**. Motivation: ① **inductive transfer** — features learned for task A help task B (related tasks regularize each other, improving generalization); ② **data efficiency** — a data-poor task borrows from a related data-rich task (a shared representation ≈ effectively more data); ③ **resource saving** — one model instead of N (on phones, in recommenders, one forward pass yields several predictions); ④ biological plausibility (one brain, many tasks). The classic architecture: **hard parameter sharing** (shared trunk + task-specific output heads). But MTL is **by no means a free lunch** — this section implements it from scratch and honestly reveals its conditions and the **negative-transfer** pitfall.

---

**中文**:**硬参数共享(hard parameter sharing)** 是最常用的 MTL 结构:
**English**: **Hard parameter sharing** is the most common MTL architecture:

```
                     ┌──→ 任务1输出头 head1 ──→ y1
输入 x ──→ 共享主干 trunk ──┤
     (学习共享表示)        └──→ 任务2输出头 head2 ──→ y2
    (shared representation)
```

**中文**:所有任务共享同一个主干(学通用特征),每个任务有自己的小输出头。训练时同时最小化各任务损失之和:$\mathcal{L}=\sum_t w_t\,\mathcal{L}_t$。共享主干因为要同时服务多个任务,被迫学到**更通用、更少过拟合**的表示——这就是 MTL 正则化效果的来源。
**English**: All tasks share one trunk (learning general features); each task has its own small head. Training minimizes the sum of task losses: $\mathcal{L}=\sum_t w_t\,\mathcal{L}_t$. Because the shared trunk must serve multiple tasks, it is forced to learn a **more general, less overfit** representation — the source of MTL's regularization effect.

**中文**:**关键前提:任务要"相关"**。如果任务共享有用的底层结构,共享主干能互相促进(**正迁移**);但如果任务无关甚至冲突,它们会**争抢主干容量、梯度方向打架**,导致所有任务都变差——这叫**负迁移(negative transfer)**。这是 MTL 最大的现实风险,本节会亲眼验证。
**English**: **Key premise: tasks must be "related."** If tasks share useful underlying structure, the shared trunk lets them reinforce each other (**positive transfer**); but if tasks are unrelated or conflicting, they **fight for trunk capacity and their gradients pull in opposing directions**, making all tasks worse — called **negative transfer**. This is MTL's biggest real-world risk, which we'll witness firsthand.

> 💡 **面试速查 / Interview cheat-sheet（★★ 深度学习/推荐系统必考）**
> **中文**:**多任务学习(MTL)**=一个模型学多个相关任务、共享表示。**为什么有用**:归纳迁移(任务互为正则)、数据高效(数据少的任务借数据多的)、省资源(一模型多输出)。**结构**:①**硬共享**(共享主干+各任务头, 最常用)；②**软共享**(各任务独立模型+参数相互约束, 如 cross-stitch)。**损失加权**是核心难点(任务量纲/难度不同): **uncertainty weighting**(Kendall, 用可学习的任务不确定度自动加权)、**GradNorm**(平衡各任务梯度范数)、**PCGrad**(投影掉冲突梯度)。**最大陷阱=负迁移**(无关/冲突任务共享→互相拖累, 全变差)→需选相关任务、调权重、或分组学(Which Tasks Together)。**MTL 不是免费午餐**——很多时候朴素硬共享打不过单任务基线, 尤其正迁移常出现在"数据多的辅助任务救数据少的主任务"。**vs 迁移学习**:迁移是"先A后B串行", MTL 是"A/B同时学"。用途:推荐(CTR+CVR+时长, 如 MMoE/ESMM)、自动驾驶(检测+分割+深度)、NLP(多任务微调)、人脸(检测+关键点+属性)。
> **English**: **Multi-task learning (MTL)** = one model learns several related tasks, sharing a representation. **Why useful**: inductive transfer (tasks regularize each other), data efficiency (data-poor tasks borrow from data-rich), resource saving (one model, many outputs). **Architectures**: ① **hard sharing** (shared trunk + task heads, most common); ② **soft sharing** (separate models with cross-constrained parameters, e.g. cross-stitch). **Loss weighting** is the core difficulty (tasks differ in scale/difficulty): **uncertainty weighting** (Kendall, learnable task uncertainties auto-weight), **GradNorm** (balance task gradient norms), **PCGrad** (project away conflicting gradients). **Biggest pitfall = negative transfer** (unrelated/conflicting tasks sharing → drag each other down) → pick related tasks, tune weights, or group tasks (Which Tasks Together). **MTL is not a free lunch** — naive hard sharing often can't beat single-task baselines; positive transfer most reliably appears when "a data-rich auxiliary task rescues a data-poor main task." **vs transfer learning**: transfer is "A then B, serial," MTL is "A and B, simultaneous." Uses: recommendation (CTR+CVR+dwell, e.g. MMoE/ESMM), self-driving (detection+segmentation+depth), NLP (multi-task fine-tuning), face (detection+landmarks+attributes).


In [ ]:

# ============================================================
# 多任务学习:共享主干 vs 单任务 / MTL shared-trunk vs single-task
# 中文:核心实验——主任务只有 25 个标注样本(数据稀缺)。给它配一个辅助任务(有 600 个样本)。
#      对比:①相关辅助任务(共享底层结构)能否救主任务? ②无关辅助任务会怎样(负迁移)?
# English: core experiment — the main task has only 25 labeled samples (scarce). Pair it with an auxiliary
#      task (600 samples). Compare: ① can a RELATED aux (sharing structure) rescue the main task?
#      ② what does an UNRELATED aux do (negative transfer)?
# ============================================================
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, matplotlib.pyplot as plt
D,H=30,64
def one_run(seed, related, n_main=25, n_aux=600):
    torch.manual_seed(seed); np.random.seed(seed)
    Wsh=torch.randn(D,6); a_main=torch.randn(6,1)                 # 共享特征生成器 + 主任务权重
    a_aux=(a_main+0.25*torch.randn(6,1)) if related else torch.randn(6,1)  # 相关=近似, 无关=随机
    feat=lambda X: torch.tanh(X@Wsh)                             # 共享的非线性底层特征 / shared features
    Xm=torch.randn(n_main,D); ym=feat(Xm)@a_main+0.1*torch.randn(n_main,1)  # 主任务:25 样本 / main: 25 samples
    Xa=torch.randn(n_aux,D);  ya=feat(Xa)@a_aux +0.1*torch.randn(n_aux,1)   # 辅助:600 样本 / aux: 600
    Xte=torch.randn(3000,D);  yte=feat(Xte)@a_main                          # 干净测试目标 / clean test
    mu,sd=ym.mean(),ym.std()                                     # 用训练统计标准化 / normalize by train stats
    ymn,yten=(ym-mu)/sd,(yte-mu)/sd; yan=(ya-ya.mean())/ya.std()
    # --- 单任务基线:只用主任务的 25 个样本 / single-task: only the 25 main samples ---
    stl=nn.Sequential(nn.Linear(D,H),nn.ReLU(),nn.Linear(H,H),nn.ReLU(),nn.Linear(H,1))
    opt=torch.optim.Adam(stl.parameters(),3e-3,weight_decay=1e-4)
    for _ in range(2000): opt.zero_grad(); F.mse_loss(stl(Xm),ymn).backward(); opt.step()
    stl_err=F.mse_loss(stl(Xte),yten).item()
    # --- 多任务:共享主干, 主任务头 + 辅助任务头 / MTL: shared trunk + main head + aux head ---
    class MTL(nn.Module):
        def __init__(s):
            super().__init__(); s.trunk=nn.Sequential(nn.Linear(D,H),nn.ReLU(),nn.Linear(H,H),nn.ReLU())
            s.head_main=nn.Linear(H,1); s.head_aux=nn.Linear(H,1)
        def forward(s,x): z=s.trunk(x); return s.head_main(z), s.head_aux(z)
    mt=MTL(); opt=torch.optim.Adam(mt.parameters(),3e-3,weight_decay=1e-4)
    for _ in range(2000):
        opt.zero_grad()
        loss=F.mse_loss(mt(Xm)[0],ymn) + F.mse_loss(mt(Xa)[1],yan)  # 两任务损失之和 / sum of task losses
        loss.backward(); opt.step()
    mtl_err=F.mse_loss(mt(Xte)[0],yten).item()
    return stl_err, mtl_err

seeds=range(6)
res={}
for related in [True,False]:
    errs=np.array([one_run(s,related) for s in seeds])           # (seeds, [stl,mtl])
    res[related]=(errs[:,0].mean(), errs[:,1].mean())
print(f"{'场景/scenario':<26}{'STL单任务(25样本)':>18}{'MTL(+辅助)':>14}{'改善':>10}")
for related,tag in [(True,"相关辅助 related aux"),(False,"无关辅助 unrelated aux")]:
    stl,mtl=res[related]; imp=100*(stl-mtl)/stl
    print(f"{tag:<26}{stl:>18.3f}{mtl:>14.3f}{imp:>+9.1f}%")
print("\n主任务测试 NMSE(越低越好)。相关辅助任务显著救主任务, 无关辅助几乎无用甚至拖累(负迁移)")


**中文**:核心结论一目了然:**相关**的辅助任务(共享底层结构 + 更多数据)把数据稀缺的主任务测试误差显著降低(正迁移);而**无关**的辅助任务几乎没有帮助,甚至轻微拖累(负迁移)。**MTL 的成败完全取决于任务是否相关**。下面可视化,并诚实讨论 MTL 更微妙的一面。
**English**: The core conclusion is clear: a **related** auxiliary task (shared structure + more data) markedly lowers the data-scarce main task's test error (positive transfer); an **unrelated** auxiliary task barely helps, even slightly hurts (negative transfer). **MTL's success hinges entirely on task relatedness.** Below we visualize and honestly discuss MTL's subtler side.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 正迁移 vs 负迁移 / positive vs negative transfer
labels=["相关辅助\nrelated aux","无关辅助\nunrelated aux"]
stl_vals=[res[True][0],res[False][0]]; mtl_vals=[res[True][1],res[False][1]]
x=np.arange(2); w=0.35
ax[0].bar(x-w/2,stl_vals,w,label="单任务 STL(仅25样本)",color="#BBBBBB")
ax[0].bar(x+w/2,mtl_vals,w,label="多任务 MTL(+辅助)",color=["#55A868","#C44E52"])
for i,(s,m) in enumerate(zip(stl_vals,mtl_vals)):
    ax[0].text(i+w/2,m+0.01,f"{100*(s-m)/s:+.0f}%",ha="center",fontsize=11,weight="bold")
ax[0].set_xticks(x); ax[0].set_xticklabels(labels); ax[0].set_ylabel("主任务测试 NMSE(越低越好)")
ax[0].set_title("正迁移(相关)vs 负迁移(无关)/ positive vs negative transfer"); ax[0].legend(fontsize=9)
# ② MTL 结构示意 / architecture
ax[1].axis("off")
ax[1].text(0.5,0.92,"硬参数共享 / hard parameter sharing",ha="center",fontsize=13,weight="bold",transform=ax[1].transAxes)
ax[1].annotate("",xy=(0.35,0.6),xytext=(0.1,0.6),arrowprops=dict(arrowstyle="->",lw=2),transform=ax[1].transAxes)
ax[1].text(0.02,0.62,"输入x",fontsize=10,transform=ax[1].transAxes)
ax[1].add_patch(plt.Rectangle((0.35,0.5),0.22,0.2,fill=True,color="#4C72B0",alpha=0.3,transform=ax[1].transAxes))
ax[1].text(0.46,0.6,"共享主干\ntrunk",ha="center",va="center",fontsize=10,transform=ax[1].transAxes)
ax[1].annotate("",xy=(0.75,0.72),xytext=(0.57,0.63),arrowprops=dict(arrowstyle="->",lw=2),transform=ax[1].transAxes)
ax[1].annotate("",xy=(0.75,0.48),xytext=(0.57,0.57),arrowprops=dict(arrowstyle="->",lw=2),transform=ax[1].transAxes)
ax[1].add_patch(plt.Rectangle((0.75,0.66),0.16,0.12,fill=True,color="#55A868",alpha=0.4,transform=ax[1].transAxes))
ax[1].add_patch(plt.Rectangle((0.75,0.42),0.16,0.12,fill=True,color="#DD8452",alpha=0.4,transform=ax[1].transAxes))
ax[1].text(0.83,0.72,"头1→y1",ha="center",va="center",fontsize=9,transform=ax[1].transAxes)
ax[1].text(0.83,0.48,"头2→y2",ha="center",va="center",fontsize=9,transform=ax[1].transAxes)
ax[1].text(0.5,0.28,"共享主干学通用特征(正则化+数据高效)\nshared trunk learns general features",ha="center",fontsize=9,style="italic",transform=ax[1].transAxes)
ax[1].text(0.5,0.15,"损失=Σ wₜ·Lₜ, 权重 wₜ 需精心平衡\nloss = Σ wₜ·Lₜ, weights need careful balancing",ha="center",fontsize=9,color="#C44E52",transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/adv11_viz.png",dpi=80); plt.show()
print("左:相关辅助任务带来正迁移(绿, 误差下降), 无关任务负迁移(红, 无改善/更差)")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **MTL 真正的威力在"数据稀缺 + 相关辅助任务"**:主任务只有 25 个标注(远不够学好),但配上一个**相关的、数据充足(600样本)**的辅助任务后,共享主干借助辅助任务的大量数据学到了**更好的底层表示**,主任务测试误差显著下降(正迁移)。这正是现实中 MTL 最常见、最有价值的用法——**用一个便宜/数据多的辅助任务,去救一个昂贵/数据少的主任务**(如:用海量点击数据的辅助任务,帮只有少量转化数据的主任务)。
2. **负迁移是真实且危险的**:换成**无关的辅助任务**,好处基本消失甚至变负——两个任务争抢主干容量、梯度方向互相打架,共享反而有害。**这推翻了"多加个任务总没坏处"的直觉**。MTL 的第一性原理是:**只有当任务共享有用的底层结构时,共享表示才有益**。选错任务组合,MTL 比单任务还差。
3. **诚实的复杂性:MTL 常常打不过单任务基线**。这是学术界公开的尴尬事实(Standley et al. 2020 等系统研究发现,朴素硬共享经常输给独立单任务模型)。原因:①**任务冲突**(梯度打架)——解法 PCGrad(投影掉冲突分量)、任务分组;②**损失不平衡**——不同任务的损失量纲/难度差很多,大损失任务会主导训练,需 **uncertainty weighting**(用可学习的任务噪声自动加权)、**GradNorm**(平衡梯度范数);③**容量竞争**——主干不够大时任务互相挤占。**所以现实里 MTL 要认真调**:选相关任务、平衡损失、必要时用 MMoE(专家混合,让不同任务软性选择不同专家)这类进阶结构——而不是天真地把几个任务塞进一个主干就期待免费提升。

**English**:
1. **MTL's real power is "data scarcity + a related auxiliary task"**: the main task has only 25 labels (far too few to learn well), but paired with a **related, data-rich (600-sample)** auxiliary task, the shared trunk uses the auxiliary's abundant data to learn a **better underlying representation**, markedly lowering the main task's test error (positive transfer). This is MTL's most common, most valuable real-world use — **use a cheap/data-rich auxiliary task to rescue an expensive/data-poor main task** (e.g., use an abundant-click auxiliary task to help a scarce-conversion main task).
2. **Negative transfer is real and dangerous**: switch to an **unrelated** auxiliary task and the benefit basically vanishes or turns negative — the two tasks fight for trunk capacity and their gradients oppose each other, so sharing becomes harmful. **This refutes the intuition that "adding a task never hurts."** MTL's first principle: **a shared representation helps only when tasks share useful underlying structure.** Pick the wrong task combination and MTL is worse than single-task.
3. **Honest complexity: MTL often can't beat single-task baselines.** This is an openly acknowledged awkward fact in academia (systematic studies like Standley et al. 2020 find naive hard sharing frequently loses to independent single-task models). Causes: ① **task conflict** (gradients fight) — fixes: PCGrad (project out conflicting components), task grouping; ② **loss imbalance** — tasks differ greatly in loss scale/difficulty, and the large-loss task dominates training, needing **uncertainty weighting** (learnable task noise auto-weights) or **GradNorm** (balance gradient norms); ③ **capacity competition** — with too small a trunk, tasks crowd each other. **So in practice MTL must be tuned seriously**: pick related tasks, balance losses, and if needed use advanced structures like MMoE (mixture-of-experts, letting tasks softly select different experts) — rather than naively cramming tasks into one trunk and expecting a free boost.

> 💼 **实战视角 / Practical angle**
> **中文**:MTL 落地:①**推荐/广告系统**是最大用户——同时预测点击率(CTR)、转化率(CVR)、观看时长等, 用 **ESMM**(解决 CVR 样本选择偏差)、**MMoE / PLE**(专家混合缓解任务冲突, 工业界标配);②**自动驾驶**(一个骨干网同时做检测+分割+深度估计, 省算力实时性);③**NLP**(多任务微调、指令微调本质是 MTL);④**CV**(人脸检测+关键点+属性)。落地要点:①**先验证任务相关性**(相关才共享, 否则负迁移)——可先看单任务表现、任务分组实验;②**损失加权最关键**——用 uncertainty weighting / GradNorm 自动平衡, 别手调死权重;③**监控每个任务**别被"总loss下降"骗了(可能一个任务涨、一个崩);④主干容量要够, 冲突严重上 MMoE/PLE 或 PCGrad;⑤**始终和单任务基线比**——MTL 不保证赢。面试金句:*"MTL 一个模型学多相关任务共享表示, 好处是归纳迁移/数据高效/省资源, 最典型是数据多的辅助任务救数据少的主任务; 但最大风险是负迁移(无关任务互相拖累), 且损失加权是核心难点(uncertainty weighting/GradNorm/PCGrad); 推荐系统用 MMoE/ESMM 是工业界标配。"*
> **English**: MTL in practice: ① **recommendation/ads systems** are the biggest users — jointly predict click-through (CTR), conversion (CVR), dwell time, using **ESMM** (fixing CVR sample-selection bias), **MMoE / PLE** (mixture-of-experts easing task conflict, an industry standard); ② **self-driving** (one backbone doing detection+segmentation+depth, saving compute for real-time); ③ **NLP** (multi-task fine-tuning; instruction tuning is essentially MTL); ④ **CV** (face detection+landmarks+attributes). Deployment keys: ① **first verify task relatedness** (share only if related, else negative transfer) — check single-task performance and task-grouping experiments; ② **loss weighting is most critical** — auto-balance with uncertainty weighting / GradNorm, don't hand-fix dead weights; ③ **monitor each task**, don't be fooled by "total loss dropping" (one task may rise while another collapses); ④ ensure enough trunk capacity, use MMoE/PLE or PCGrad for heavy conflict; ⑤ **always compare against single-task baselines** — MTL isn't guaranteed to win. Interview line: *"MTL learns several related tasks in one model sharing a representation; benefits are inductive transfer / data efficiency / resource saving, most typically a data-rich auxiliary task rescuing a data-poor main task; but the biggest risk is negative transfer (unrelated tasks drag each other), and loss weighting is the core difficulty (uncertainty weighting/GradNorm/PCGrad); recommenders use MMoE/ESMM as an industry standard."*

---
### 小结 / Summary
- **中文**:MTL=一个模型学多相关任务共享表示(硬共享:共享主干+各任务头); 好处归纳迁移/数据高效/省资源。
- **English**: MTL = one model learns several related tasks sharing a representation (hard sharing: shared trunk + task heads); benefits inductive transfer/data efficiency/resource saving.
- **中文**:正迁移最典型=数据多的相关辅助任务救数据少的主任务(本例 +10%); 无关任务→负迁移(拖累)。
- **English**: Positive transfer most typically = a data-rich related auxiliary rescues a data-poor main task (+10% here); unrelated tasks → negative transfer (drag down).
- **中文**:MTL 非免费午餐(常输单任务基线); 核心难点损失加权(uncertainty weighting/GradNorm/PCGrad); 工业界 MMoE/ESMM。
- **English**: MTL isn't a free lunch (often loses to single-task); core difficulty is loss weighting (uncertainty weighting/GradNorm/PCGrad); industry uses MMoE/ESMM.
